# Assistente de Investimentos: RPA (Python) + N8N

Este notebook é o "robô" do projeto. Ele faz o que uma pessoa faria à mão:

1. abre a [página de clientes](https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n) (GitHub Pages);
2. lê a tabela `#clientes` (nome, email, saldo e perfil);
3. envia a lista para o Webhook do N8N, que cruza cada perfil com o [CSV de investimentos](https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n/data.csv) e devolve uma mensagem por cliente.

**Dois modos de execução:**

| `N8N_WEBHOOK` | O que acontece |
|---|---|
| preenchido | envia os clientes ao N8N e mostra as mensagens que o workflow devolveu |
| vazio | **modo simulado**: aplica aqui mesmo as mesmas regras do workflow (`n8n/workflow.json`), para testar sem N8N no ar |

## Dependências

In [ ]:
%pip install --quiet requests beautifulsoup4

## Configuração

In [2]:
import os

URL_PAGINA = "https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n/"
URL_CSV = "https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n/data.csv"

# Cole aqui a URL do nó Webhook do N8N (ex.: "https://SEU_N8N/webhook/clientes").
# No editor do N8N, a URL de teste usa /webhook-test/; com o workflow ativo, /webhook/.
# Vazio = modo simulado.
N8N_WEBHOOK = os.environ.get("N8N_WEBHOOK", "")

TIMEOUT = 30  # segundos

## Extração dos clientes (RPA)

In [3]:
import requests
from bs4 import BeautifulSoup

CAMPOS = ["nome", "email", "saldo", "perfil"]


def extrair_clientes(url: str) -> list[dict]:
    """Baixa a página e devolve um dicionário por linha da tabela #clientes."""
    resposta = requests.get(url, timeout=TIMEOUT)
    resposta.raise_for_status()
    resposta.encoding = "utf-8"

    soup = BeautifulSoup(resposta.text, "html.parser")
    linhas = soup.select("#clientes tbody tr")
    if not linhas:
        raise ValueError("Tabela #clientes não encontrada ou vazia: a página mudou?")

    clientes = []
    for linha in linhas:
        colunas = linha.find_all("td")
        cliente = {}
        for i, campo in enumerate(CAMPOS):
            # Prefere a classe da célula (td.nome, td.email...); se faltar, usa a posição.
            celula = linha.select_one(f"td.{campo}") or (colunas[i] if i < len(colunas) else None)
            cliente[campo] = celula.get_text(strip=True) if celula else ""
        clientes.append(cliente)
    return clientes


clientes = extrair_clientes(URL_PAGINA)
print(f"Clientes encontrados: {len(clientes)}")
for cliente in clientes:
    print(cliente)

Clientes encontrados: 10
{'nome': 'Ana Silva', 'email': 'ana@email.com', 'saldo': 'R$ 12.500,00', 'perfil': 'Conservador'}
{'nome': 'Bruno Lima', 'email': 'bruno@email.com', 'saldo': 'R$ 3.200,00', 'perfil': 'Moderado'}
{'nome': 'Carla Souza', 'email': 'carla@email.com', 'saldo': 'R$ 28.900,00', 'perfil': 'Arrojado'}
{'nome': 'Daniel Oliveira', 'email': 'daniel@email.com', 'saldo': 'R$ 850,00', 'perfil': 'Conservador'}
{'nome': 'Elena Rodrigues', 'email': 'elena@email.com', 'saldo': 'R$ 45.000,00', 'perfil': 'Arrojado'}
{'nome': 'Felipe Santos', 'email': 'felipe@email.com', 'saldo': 'R$ 7.800,00', 'perfil': 'Moderado'}
{'nome': 'Gabriela Costa', 'email': 'gabriela@email.com', 'saldo': 'R$ 1.500,00', 'perfil': 'Conservador'}
{'nome': 'Henrique Alves', 'email': 'henrique@email.com', 'saldo': 'R$ 22.300,00', 'perfil': 'Moderado'}
{'nome': 'Isabela Martins', 'email': 'isabela@email.com', 'saldo': 'R$ 500,00', 'perfil': 'Conservador'}
{'nome': 'João Pereira', 'email': 'joao@email.com', 'sal

## Regras do cruzamento (as mesmas do nó Code do N8N)

- o saldo chega como texto (`"R$ 12.500,00"`) e é convertido para número;
- o perfil é comparado sem diferenciar maiúsculas, acentos e espaços;
- só é recomendado o produto do perfil cujo **mínimo cabe no saldo**; os que não cabem aparecem como "faltam R$ X";
- perfil desconhecido ou saldo ilegível geram uma mensagem pedindo atualização do cadastro, em vez de uma recomendação errada.

In [4]:
import csv
import io
import unicodedata

ABERTURA = {
    "conservador": "Como seu perfil é Conservador, priorizamos renda fixa, com foco em segurança e liquidez.",
    "moderado": "Como seu perfil é Moderado, sugerimos equilibrar proteção e crescimento, combinando prefixado, inflação e multimercado.",
    "arrojado": "Como seu perfil é Arrojado, destacamos renda variável, com maior potencial de retorno e também mais oscilação.",
}
AVISO = "Dados fictícios, para fins educacionais. Não é recomendação de investimento."


def normalizar(texto) -> str:
    texto = unicodedata.normalize("NFD", str(texto or ""))
    return "".join(c for c in texto if unicodedata.category(c) != "Mn").strip().lower()


def ler_valor(texto):
    """'R$ 12.500,00' -> 12500.0; None se não houver número."""
    limpo = "".join(c for c in str(texto or "") if c.isdigit() or c in ",.-")
    limpo = limpo.replace(".", "").replace(",", ".", 1)
    try:
        return float(limpo)
    except ValueError:
        return None


def reais(valor: float) -> str:
    return "R$ " + f"{valor:,.2f}".replace(",", "_").replace(".", ",").replace("_", ".")


def rentabilidade(texto: str) -> str:
    if normalizar(texto).startswith("variavel"):
        return "rentabilidade variável"
    texto = texto.replace("+", " + ")
    return "".join(
        "," if c == "." and 0 < i < len(texto) - 1 and texto[i - 1].isdigit() and texto[i + 1].isdigit() else c
        for i, c in enumerate(texto)
    )


def listar(itens: list[str]) -> str:
    return itens[0] if len(itens) == 1 else ", ".join(itens[:-1]) + " e " + itens[-1]


def ler_produtos(csv_texto: str) -> list[dict]:
    leitor = csv.DictReader(io.StringIO(csv_texto.strip()))
    leitor.fieldnames = [normalizar(c) for c in leitor.fieldnames]
    return [
        {"perfil": l["perfil"].strip(), "produto": l["produto"].strip(),
         "minimo": float(l["minimo"]), "rentabilidade": l["rentabilidade"].strip()}
        for l in leitor
    ]


def gerar_mensagem(cliente: dict, produtos: list[dict]) -> dict:
    nome = str(cliente.get("nome") or "").strip()
    primeiro_nome = nome.split()[0] if nome else "cliente"
    chave = normalizar(cliente.get("perfil"))
    saldo = ler_valor(cliente.get("saldo"))
    base = {"nome": nome, "email": str(cliente.get("email") or "").strip(),
            "perfil": str(cliente.get("perfil") or "").strip(), "saldo": saldo}

    if chave not in ABERTURA:
        return {**base, "status": "perfil_invalido", "recomendados": [], "fora_do_alcance": [],
                "mensagem": f"Olá, {primeiro_nome}! Não encontramos um perfil de investidor válido no seu cadastro. "
                            "Atualize seu questionário de perfil para receber recomendações."}
    if saldo is None:
        return {**base, "status": "saldo_invalido", "recomendados": [], "fora_do_alcance": [],
                "mensagem": f"Olá, {primeiro_nome}! Não conseguimos ler o saldo da sua conta. "
                            "Fale com a gente para atualizar seus dados."}

    do_perfil = [p for p in produtos if normalizar(p["perfil"]) == chave]
    cabem = [p for p in do_perfil if p["minimo"] <= saldo]
    nao_cabem = [p for p in do_perfil if p["minimo"] > saldo]

    texto = f"Olá, {primeiro_nome}! {ABERTURA[chave]}"
    if cabem:
        itens = [f"{p['produto']} ({rentabilidade(p['rentabilidade'])}, a partir de {reais(p['minimo'])})" for p in cabem]
        texto += f" Com o seu saldo de {reais(saldo)}, você já pode aplicar em: {listar(itens)}."
    else:
        texto += f" Com o seu saldo de {reais(saldo)}, ainda não há opção disponível para o seu perfil."
    for p in nao_cabem:
        texto += f" Faltam {reais(p['minimo'] - saldo)} para {p['produto']} (mínimo de {reais(p['minimo'])})."
    texto += " Conte com a gente para montar sua carteira!"

    return {**base, "status": "ok" if cabem else "sem_produto_no_saldo",
            "recomendados": [p["produto"] for p in cabem],
            "fora_do_alcance": [p["produto"] for p in nao_cabem],
            "mensagem": texto}


def gerar_localmente(clientes: list[dict], csv_texto: str) -> dict:
    produtos = ler_produtos(csv_texto)
    mensagens = [gerar_mensagem(c, produtos) for c in clientes]
    return {"total": len(mensagens), "aviso": AVISO, "mensagens": mensagens}

## Enviar ao N8N (ou simular)

In [5]:
if N8N_WEBHOOK:
    print(f"Enviando {len(clientes)} clientes para o N8N...")
    resposta = requests.post(N8N_WEBHOOK, json={"clientes": clientes}, timeout=TIMEOUT)
    resposta.raise_for_status()
    resultado = resposta.json()
    origem = "N8N"
else:
    print("N8N_WEBHOOK vazio: modo simulado (mesmas regras do workflow).")
    csv_resposta = requests.get(URL_CSV, timeout=TIMEOUT)
    csv_resposta.raise_for_status()
    csv_resposta.encoding = "utf-8"
    resultado = gerar_localmente(clientes, csv_resposta.text)
    origem = "simulado"

print(f"\n{resultado['total']} mensagens ({origem}). {resultado['aviso']}\n")
for m in resultado["mensagens"]:
    print(f"Para: {m['nome']} <{m['email']}> [{m['perfil']} | {m['status']}]")
    print(m["mensagem"], end="\n\n")

Enviando 10 clientes para o N8N...



10 mensagens (N8N). Dados fictícios, para fins educacionais. Não é recomendação de investimento.

Para: Ana Silva <ana@email.com> [Conservador | ok]
Olá, Ana! Como seu perfil é Conservador, priorizamos renda fixa, com foco em segurança e liquidez. Com o seu saldo de R$ 12.500,00, você já pode aplicar em: Poupança (6,2%, a partir de R$ 100,00), CDB Liquidez Diária (12,5%, a partir de R$ 1.000,00) e Tesouro Selic (13,0%, a partir de R$ 500,00). Conte com a gente para montar sua carteira!

Para: Bruno Lima <bruno@email.com> [Moderado | ok]
Olá, Bruno! Como seu perfil é Moderado, sugerimos equilibrar proteção e crescimento, combinando prefixado, inflação e multimercado. Com o seu saldo de R$ 3.200,00, você já pode aplicar em: CDB Prefixado (14,0%, a partir de R$ 1.000,00) e Tesouro IPCA+ (IPCA + 6,0%, a partir de R$ 1.000,00). Faltam R$ 1.800,00 para Fundo Multimercado (mínimo de R$ 5.000,00). Conte com a gente para montar sua carteira!

Para: Carla Souza <carla@email.com> [Arrojado | ok]